
## Important Note on Milestone Deliverables

For milestones M1–M3, full publication-style prose is neither expected nor recommended. To strongly point to the main outcome of these milestone publication-style prose is not allowed in the age of LLMs. 

These milestones are intended to document scientific understanding, technical analysis, contextualization, and critical reasoning and not polished final writing.

Students are therefore encouraged to use:
- bullet points,
- concise technical statements,
- structured notes,
- comparison tables,
- diagrams,
- pseudocode,
- formulas,
- annotations,
- and partially structured LaTeX sections.

Deliverables should already use the final paper template (or if not already known a standard scientific LaTeX template when no template is provided) in order to incrementally develop the final scientific artifact.

The focus of the early milestones is scientific reasoning and engineering analysis, not rhetorical polishing.


# Seminar Work Milestone Checklist

This notebook supports the structured preparation of a seminar work in computer science / embedded systems.

The seminar starts from one assigned reference paper or book chapter. The final outputs are a scientific paper and a presentation. However, the main learning objective is not only the final text, but the stepwise development of scientific understanding, contextualization, critical evaluation, and communication competence.

Use this notebook as a checklist before submitting each milestone. It is not a replacement for the milestone artifacts themselves.


In [1]:
import json
import re
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError as e:
    raise ImportError("ipywidgets is required. Install with: pip install ipywidgets") from e


PROTOCOL_VERSION = "0.2"

verification_levels = [
    "none",
    "plausibility-check",
    "secondary-source-check",
    "primary-source-check",
    "methodological-check",
    "empirical-check",
    "multi-source-validation"
]

evaluation_statuses = [
    "accepted",
    "partially-accepted",
    "rejected",
    "revised",
    "unresolved"
]

usefulness_levels = [
    "low",
    "medium",
    "high"
]

usage_types = [
    "idea-generation",
    "literature-search-support",
    "summarization",
    "terminology-clarification",
    "comparison-structuring",
    "argument-critique",
    "text-revision",
    "translation",
    "code-generation",
    "verification-support",
    "modeling-support",
    "requirements engineering-support",
    "other"
]

metadata = {
    "protocol_version": PROTOCOL_VERSION,
    "student_id": "",
    "course": "",
    "milestone": "",
    "topic": "",
    "entries": []
}


student_id = widgets.Text(
    description="Student ID:",
    placeholder="anonymous or matriculation-compatible ID"
)

course = widgets.Text(
    description="Course:",
    placeholder="Course/module name"
)

milestone = widgets.Text(
    description="Milestone:",
    placeholder="e.g. literature-analysis"
)

topic = widgets.Text(
    description="Topic:",
    placeholder="Your seminar/lab topic"
)

metadata_box = widgets.VBox([
    widgets.HTML("<h3>Protocol Metadata</h3>"),
    student_id,
    course,
    milestone,
    topic
])

entries_box = widgets.VBox([])
status_output = widgets.Output()


def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def make_interaction_widget(parent_id=None, entry_id=None):
    if entry_id is None:
        entry_id = f"e-{uuid.uuid4().hex[:8]}"

    objective = widgets.Textarea(
        description="Objective:",
        placeholder="What scientific task did you try to solve?",
        layout=widgets.Layout(width="100%", height="70px")
    )

    ai_tool = widgets.Text(
        description="AI tool:",
        placeholder="e.g. ChatGPT, Claude, Perplexity"
    )

    ai_model = widgets.Text(
        description="Model:",
        placeholder="e.g. GPT-5.5, unknown"
    )

    prompt = widgets.Textarea(
        description="Prompt:",
        placeholder="Paste the relevant prompt here.",
        layout=widgets.Layout(width="100%", height="120px")
    )

    ai_output_summary = widgets.Textarea(
        description="AI output summary:",
        placeholder="Summarize the AI output. Do not paste long raw outputs unless required.",
        layout=widgets.Layout(width="100%", height="100px")
    )

    verification_level = widgets.RadioButtons(
        options=verification_levels,
        description="Verification level:",
        value="plausibility-check"
    )

    verification_actions = widgets.Textarea(
        description="Verification actions:",
        placeholder="What exactly did you check, and against which sources/evidence?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    evaluation_status = widgets.RadioButtons(
        options=evaluation_statuses,
        description="Status:",
        value="partially-accepted"
    )

    usefulness = widgets.RadioButtons(
        options=usefulness_levels,
        description="Usefulness:",
        value="medium"
    )

    issues = widgets.Textarea(
        description="Issues:",
        placeholder="List hallucinations, overgeneralizations, missing assumptions, wrong citations, etc.",
        layout=widgets.Layout(width="100%", height="90px")
    )

    used_in_work = widgets.Checkbox(
        description="Used in submitted work",
        value=False
    )

    section = widgets.Text(
        description="Section:",
        placeholder="e.g. Section 2.3"
    )

    usage_type = widgets.Dropdown(
        options=usage_types,
        description="Usage type:",
        value="other"
    )

    direct_text_reused = widgets.Checkbox(
        description="Direct text reused",
        value=False
    )

    reflection = widgets.Textarea(
        description="Reflection:",
        placeholder="What did you learn? How did this affect your understanding or scientific decision?",
        layout=widgets.Layout(width="100%", height="100px")
    )

    remove_button = widgets.Button(
        description="Remove interaction",
        button_style="danger"
    )

    box = widgets.VBox([
        widgets.HTML(f"<hr><h3>Interaction {entry_id}</h3>"),
        objective,
        widgets.HBox([ai_tool, ai_model]),
        prompt,
        ai_output_summary,
        widgets.HTML("<b>Verification</b>"),
        verification_level,
        verification_actions,
        widgets.HTML("<b>Evaluation</b>"),
        evaluation_status,
        usefulness,
        issues,
        widgets.HTML("<b>Integration</b>"),
        used_in_work,
        section,
        usage_type,
        direct_text_reused,
        widgets.HTML("<b>Reflection</b>"),
        reflection,
        remove_button
    ])

    box.entry_id = entry_id
    box.parent_id = parent_id
    box.fields = {
        "objective": objective,
        "ai_tool": ai_tool,
        "ai_model": ai_model,
        "prompt": prompt,
        "ai_output_summary": ai_output_summary,
        "verification_level": verification_level,
        "verification_actions": verification_actions,
        "evaluation_status": evaluation_status,
        "usefulness": usefulness,
        "issues": issues,
        "used_in_work": used_in_work,
        "section": section,
        "usage_type": usage_type,
        "direct_text_reused": direct_text_reused,
        "reflection": reflection
    }

    def remove_entry(_):
        children = list(entries_box.children)
        if box in children:
            children.remove(box)
            entries_box.children = tuple(children)

    remove_button.on_click(remove_entry)
    return box


def add_interaction(_):
    new_box = make_interaction_widget()
    entries_box.children = tuple(list(entries_box.children) + [new_box])


def serialize_entry(box):
    f = box.fields

    verification_actions = [
        line.strip()
        for line in f["verification_actions"].value.splitlines()
        if line.strip()
    ]

    issues = [
        line.strip()
        for line in f["issues"].value.splitlines()
        if line.strip()
    ]

    return {
        "id": box.entry_id,
        "timestamp": now_iso(),
        "parent_id": box.parent_id,
        "objective": f["objective"].value.strip(),
        "ai_tool": {
            "name": f["ai_tool"].value.strip(),
            "model": f["ai_model"].value.strip()
        },
        "prompt": f["prompt"].value.strip(),
        "ai_output_summary": f["ai_output_summary"].value.strip(),
        "verification": {
            "actions": verification_actions,
            "level": f["verification_level"].value
        },
        "evaluation": {
            "status": f["evaluation_status"].value,
            "issues": issues,
            "usefulness": f["usefulness"].value
        },
        "integration": {
            "used_in_work": bool(f["used_in_work"].value),
            "section": f["section"].value.strip(),
            "usage_type": f["usage_type"].value,
            "direct_text_reused": bool(f["direct_text_reused"].value)
        },
        "reflection": f["reflection"].value.strip()
    }


def safe_bibtex_key(text):
    text = (text or "ai-usage-protocol").lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text or "ai-usage-protocol"


def escape_bibtex(value):
    value = str(value or "")
    return (
        value.replace("\\", "\\textbackslash{}")
             .replace("{", "\\{")
             .replace("}", "\\}")
             .replace("&", "\\&")
             .replace("%", "\\%")
             .replace("$", "\\$")
             .replace("#", "\\#")
             .replace("_", "\\_")
    )


def protocol_to_bibtex(protocol, json_filename=None):
    exported_at = protocol.get("exported_at") or now_iso()
    year = exported_at[:4]
    student = protocol.get("student_id") or "anonymous"
    course_name = protocol.get("course") or "Course"
    milestone_name = protocol.get("milestone") or "Milestone"
    topic_name = protocol.get("topic") or "AI-assisted scientific work"
    version = protocol.get("protocol_version") or PROTOCOL_VERSION
    entries_count = len(protocol.get("entries", []))
    json_file = json_filename or "ai_usage_protocol.json"

    key_parts = [
        "ai-usage-protocol",
        safe_bibtex_key(student),
        safe_bibtex_key(milestone_name),
        year,
    ]

    bib_key = "-".join([part for part in key_parts if part])

    title = f"AI Usage Protocol for {milestone_name}: {topic_name}"

    note = (
        f"Machine-readable documentation of AI-assisted scientific work; "
        f"course: {course_name}; protocol version: {version}; "
        f"documented reasoning episodes: {entries_count}; "
        f"JSON file: {json_file}"
    )

    return f"""@misc{{{bib_key},
  author = {{{escape_bibtex(student)}}},
  title = {{{escape_bibtex(title)}}},
  year = {{{escape_bibtex(year)}}},
  howpublished = {{{escape_bibtex("Submitted AI usage protocol notebook and JSON file")}}},
  note = {{{escape_bibtex(note)}}}
}}"""


def save_protocol(_):
    metadata["protocol_version"] = PROTOCOL_VERSION
    metadata["student_id"] = student_id.value.strip()
    metadata["course"] = course.value.strip()
    metadata["milestone"] = milestone.value.strip()
    metadata["topic"] = topic.value.strip()
    metadata["exported_at"] = now_iso()
    metadata["entries"] = [serialize_entry(box) for box in entries_box.children]

    base_filename = (
        f"ai_usage_protocol_"
        f"{metadata['student_id'] or 'anonymous'}_"
        f"{metadata['milestone'] or 'milestone'}"
    )

    base_filename = base_filename.replace(" ", "_").replace("/", "-")
    json_filename = f"{base_filename}.json"
    bib_filename = f"{base_filename}.bib"

    Path(json_filename).write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    bibtex_entry = protocol_to_bibtex(
        metadata,
        json_filename=json_filename
    )

    Path(bib_filename).write_text(
        bibtex_entry + "\n",
        encoding="utf-8"
    )

    with status_output:
        clear_output()
        print(f"Saved protocol JSON: {json_filename}")
        print(f"Saved BibTeX entry: {bib_filename}")
        print(f"Entries: {len(metadata['entries'])}")
        print()
        print("BibTeX entry:")
        print(bibtex_entry)


def load_protocol_from_json(_):
    json_files = list(Path(".").glob("ai_usage_protocol_*.json"))

    with status_output:
        clear_output()

        if not json_files:
            print("No ai_usage_protocol_*.json file found.")
            print("First upload your previous protocol JSON file on the left Files panel.")
            return

        print("Available protocol JSON files:")
        for file in json_files:
            print("-", file.name)

        # Load the newest protocol JSON file
        protocol_file = max(json_files, key=lambda p: p.stat().st_mtime)

        try:
            loaded = json.loads(protocol_file.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Could not load JSON file: {protocol_file.name}")
            print(e)
            return

        metadata.clear()
        metadata.update(loaded)

        student_id.value = loaded.get("student_id", "")
        course.value = loaded.get("course", "")
        milestone.value = loaded.get("milestone", "")
        topic.value = loaded.get("topic", "")

        loaded_boxes = []

        for entry in loaded.get("entries", []):
            box = make_interaction_widget(
                parent_id=entry.get("parent_id"),
                entry_id=entry.get("id")
            )

            f = box.fields

            f["objective"].value = entry.get("objective", "")

            ai_tool_data = entry.get("ai_tool", {})
            f["ai_tool"].value = ai_tool_data.get("name", "")
            f["ai_model"].value = ai_tool_data.get("model", "")

            f["prompt"].value = entry.get("prompt", "")
            f["ai_output_summary"].value = entry.get("ai_output_summary", "")

            verification_data = entry.get("verification", {})
            verification_level_value = verification_data.get(
                "level",
                "plausibility-check"
            )

            if verification_level_value in verification_levels:
                f["verification_level"].value = verification_level_value

            f["verification_actions"].value = "\n".join(
                verification_data.get("actions", [])
            )

            evaluation_data = entry.get("evaluation", {})
            evaluation_status_value = evaluation_data.get(
                "status",
                "partially-accepted"
            )

            if evaluation_status_value in evaluation_statuses:
                f["evaluation_status"].value = evaluation_status_value

            usefulness_value = evaluation_data.get("usefulness", "medium")

            if usefulness_value in usefulness_levels:
                f["usefulness"].value = usefulness_value

            f["issues"].value = "\n".join(
                evaluation_data.get("issues", [])
            )

            integration_data = entry.get("integration", {})
            f["used_in_work"].value = bool(
                integration_data.get("used_in_work", False)
            )

            f["section"].value = integration_data.get("section", "")

            usage_type_value = integration_data.get("usage_type", "other")

            if usage_type_value in usage_types:
                f["usage_type"].value = usage_type_value

            f["direct_text_reused"].value = bool(
                integration_data.get("direct_text_reused", False)
            )

            f["reflection"].value = entry.get("reflection", "")

            loaded_boxes.append(box)

        entries_box.children = tuple(loaded_boxes)

        print()
        print(f"Loaded protocol JSON: {protocol_file.name}")
        print(f"Loaded entries: {len(loaded_boxes)}")
        print()
        print("Now you can click 'Add interaction' to add new interactions.")
        print("After adding new interactions, click 'Save protocol JSON'.")


add_button = widgets.Button(
    description="Add interaction",
    button_style="primary"
)

save_button = widgets.Button(
    description="Save protocol JSON",
    button_style="success"
)

load_button = widgets.Button(
    description="Load previous JSON",
    button_style="warning"
)

add_button.on_click(add_interaction)
save_button.on_click(save_protocol)
load_button.on_click(load_protocol_from_json)

display(metadata_box)
display(widgets.HBox([load_button, add_button, save_button]))
display(entries_box)
display(status_output)

VBox()

Output()

## Milestone 1: Technical Understanding and Research Framing

### Scientific purpose
The first milestone ensures that the assigned paper or book chapter has been technically understood before broader analysis or text production starts. In an AI-supported workflow, this milestone is especially important because a plausible summary can be generated without real understanding. The relevant competence is therefore not producing a polished summary, but identifying the technical contribution, assumptions, problem setting, and open points.

### Expected output
A short structured technical briefing, typically 2–4 pages, including:

- problem statement and motivation
- core technical idea
- relevant formalism, algorithm, architecture, or model
- assumptions and scope limitations
- runtime, memory, scalability, or resource aspects, if applicable
- unclear points and questions for feedback


In [2]:
m1_items = [
    "I can state the technical problem without copying the abstract.",
    "I can explain the main contribution in my own words.",
    "I identified the central algorithm, formalism, model, or architecture.",
    "I identified assumptions and scope limitations.",
    "I considered runtime, memory usage, scalability, or resource constraints where applicable.",
    "I documented unclear points instead of hiding them.",
    "I can explain why the work is relevant for embedded systems / computer science."
]

m1_checks = [widgets.Checkbox(description=x, indent=False) for x in m1_items]
m1_notes = widgets.Textarea(description="Open questions:", layout=widgets.Layout(width="100%", height="100px"))
display(widgets.HTML("<b>M1 checklist</b>"))
display(widgets.VBox(m1_checks + [m1_notes]))


HTML(value='<b>M1 checklist</b>')

## Milestone 2: Scientific Contextualization and Tradeoff Analysis

### Scientific purpose
This milestone replaces a purely classical “related work” task with a broader scientific and engineering contextualization. Some seminar topics, especially in scheduling, formal methods, or embedded systems, may be driven by one central author, one framework, or one line of work. In such cases, the task is not to artificially collect many weakly related papers. Instead, students should identify related approaches, neighboring problem classes, design alternatives, assumptions, and consequences.

The goal is to understand where the assigned work sits in the scientific and engineering landscape.

### Expected output
A structured context and tradeoff analysis including, where applicable:

- related papers, approaches, methods, tools, standards, or paradigms
- competing or neighboring solution strategies
- assumptions behind different approaches
- tradeoffs such as precision vs runtime, memory vs accuracy, optimality vs feasibility, offline vs online computation, or formal guarantees vs practical deployability
- consequences for embedded systems, real-time systems, cyber-physical systems, or constrained hardware
- explanation of why some related work is missing, sparse, or only indirectly related


In [8]:
m2_items = [
    "I did not only summarize related papers, but compared approaches or assumptions.",
    "I identified at least one relevant neighboring approach, paradigm, or design alternative.",
    "I explained whether the topic has strong, weak, or sparse related work and why.",
    "I analyzed scientific or engineering tradeoffs.",
    "I considered embedded systems constraints such as timing, memory, energy, determinism, or deployment context.",
    "I can justify why selected references or approaches are relevant.",
    "I can justify why excluded references or approaches are less relevant."
]

m2_checks = [import ipywidgets as widgets.Checkbox(description=x, indent=False) for x in m2_items]
m2_notes = widgets.Textarea(description="Key tradeoffs / context notes:", layout=widgets.Layout(width="100%", height="120px"))
display(widgets.HTML("<b>M2 checklist</b>"))
display(widgets.VBox(m2_checks + [m2_notes]))


SyntaxError: invalid syntax (3154081569.py, line 11)

## Milestone 3: Critical Evaluation and Transfer

### Scientific purpose
This milestone evaluates whether the student can move beyond understanding and contextualization toward scientific judgment. In an AI-rich environment, this is one of the most important seminar competences: students must be able to assess the strengths, weaknesses, assumptions, risks, and applicability of a technical contribution.

The focus is not whether the original paper is “good” or “bad”, but under which assumptions it is useful, where it may fail, and how it transfers to realistic embedded systems contexts.

### Expected output
A critical evaluation report including:

- strengths of the approach
- limitations and risks
- assumptions that may be unrealistic or restrictive
- runtime, memory, scalability, or implementation implications
- application examples where the approach is suitable
- application examples where the approach is unsuitable
- possible extensions, adaptations, or open research questions


In [3]:
m3_items = [
    "I identified concrete strengths, not only generic advantages.",
    "I identified concrete limitations, risks, or hidden assumptions.",
    "I considered whether the approach scales in runtime, memory, or system size.",
    "I evaluated applicability to realistic embedded systems scenarios.",
    "I identified at least one case where the approach is suitable.",
    "I identified at least one case where the approach is unsuitable or problematic.",
    "I proposed a meaningful extension, adaptation, or open question."
]

m3_checks = [widgets.Checkbox(description=x, indent=False) for x in m3_items]
m3_notes = widgets.Textarea(description="Critical evaluation notes:", layout=widgets.Layout(width="100%", height="120px"))
display(widgets.HTML("<b>M3 checklist</b>"))
display(widgets.VBox(m3_checks + [m3_notes]))


HTML(value='<b>M3 checklist</b>')

## Milestone 4: Scientific Communication: Paper and Talk

### Scientific purpose
The final paper and talk synthesize the previous milestones into coherent scientific communication. In this seminar model, the final paper is not the only or primary evidence of competence, because final prose can be strongly supported by LLMs. Its role is to document and communicate the understanding, contextualization, and evaluation developed in the milestones.

The presentation should not simply repeat the paper. It should communicate the central technical idea, the key tradeoffs, and the critical assessment clearly to an expert audience.

### Expected output

#### Final paper
- coherent explanation of the technical contribution
- scientific context and related approaches/tradeoffs
- critical evaluation and application discussion
- clear citations and transparent AI usage protocol, if required

#### Talk/slides
- motivation and problem setting
- core technical idea using diagrams, examples, or formal sketches
- main assumptions and tradeoffs
- strengths, limitations, and application examples
- discussion questions or open issues


In [6]:
m4_items = [
    "The paper integrates M1–M3 instead of being generated independently at the end.",
    "The technical explanation is understandable without oversimplifying the contribution.",
    "The paper clearly distinguishes facts, interpretation, critique, and speculation.",
    "The slides focus on explanation and discussion, not on text-heavy reproduction of the paper.",
    "The presentation includes examples, diagrams, or formal sketches where helpful.",
    "The talk addresses strengths, limitations, and application consequences.",
    "AI usage is documented according to the required protocol."
]

m4_checks = [widgets.Checkbox(description=x, indent=False) for x in m4_items]
m4_notes = widgets.Textarea(description="Communication notes:", layout=widgets.Layout(width="100%", height="120px"))
display(widgets.HTML("<b>M4 checklist</b>"))
display(widgets.VBox(m4_checks + [m4_notes]))


HTML(value='<b>M4 checklist</b>')

## In-Class Competency Validation

The milestones may be accompanied by short in-class scientific tasks. These tasks are not intended to reproduce the submitted text. They validate whether the student can reason about their own milestone work under controlled conditions.

Possible task types:

- explain a technical assumption from the assigned work
- compare two approaches or design alternatives
- critique a scalability or runtime claim
- evaluate whether an approach fits a specific embedded systems scenario
- identify an unsupported or overgeneralized claim
- justify why a reference, method, or application example was included or excluded
- revise or critique an AI-generated explanation related to the milestone

The goal is authenticated scientific understanding, not memorization.
